모듈 5는 GR00T용 인프라(ECR/CodeBuild/Batch/SageMaker/MLflow)를 배포하고, fine-tune 전에 base 모델이 GR00T Policy Server에서 정상 추론되는지 스모크 테스트합니다.

## 1단계: 인프라 배포 (CDK)

이미 배포됐으면 no-op입니다. 관리자 1회 `deploy:shared` → 사용자별 `deploy`.

In [ ]:
USER_ID = "alice"   # 본인 식별자로 변경
REGION = "us-east-1"
!cd ../../infra/groot && npm install --silent && npm run deploy:shared
!cd ../../infra/groot && npm run deploy -- -c userId={USER_ID} -c region={REGION}
!cd ../../infra/groot && npx ts-node bin/update-config.ts --user-id {USER_ID} --region {REGION}

## 2단계: config 확인

In [ ]:
from pathlib import Path
import yaml
CONFIG = yaml.safe_load((Path.cwd().parent / "config.yaml").read_text())
CONFIG["aws"], CONFIG.get("ecr", {})

## 3단계: GR00T Policy Server 검증 (ZMQ)

서버는 별도로 떠 있어야 합니다(수동 빌드·실행 절차는 `infra/isaaclab/documents/3-groot-verification-guide.md` 참고). REQ 소켓으로 `ping` 후 더미 `get_action` 요청을 보내 응답 shape을 확인합니다.

In [ ]:
import zmq, msgpack
SERVER_IP = "127.0.0.1"   # 원격이면 인스턴스 IP
ctx = zmq.Context(); sock = ctx.socket(zmq.REQ); sock.setsockopt(zmq.RCVTIMEO, 10000)
sock.connect(f"tcp://{SERVER_IP}:5555")
sock.send(msgpack.packb({"endpoint": "ping"}))
print("ping 응답:", msgpack.unpackb(sock.recv()))

## 4단계: 더미 observation으로 get_action 호출 (GR1 임베디먼트)

`groot/inference/batch-zmq/test_inference_remote.py`와 동일한 방식으로 numpy 배열을 `.npy` 바이트로 감싸 msgpack으로 인코딩/디코딩합니다.

In [ ]:
import numpy as np, io

def encode_ndarray(obj):
    if isinstance(obj, np.ndarray):
        buf = io.BytesIO()
        np.save(buf, obj, allow_pickle=False)
        return {"__ndarray_class__": True, "as_npy": buf.getvalue()}
    return obj

def decode_ndarray(obj):
    if isinstance(obj, dict) and "__ndarray_class__" in obj:
        return np.load(io.BytesIO(obj["as_npy"]), allow_pickle=False)
    return obj

In [ ]:
# 더미 observation (GR1 임베디먼트)
observation = {
    "video": {
        "ego_view_bg_crop_pad_res256_freq20": np.random.randint(0, 255, (1, 1, 256, 256, 3), dtype=np.uint8)
    },
    "state": {
        "left_arm": np.random.rand(1, 1, 7).astype(np.float32),
        "right_arm": np.random.rand(1, 1, 7).astype(np.float32),
        "left_hand": np.random.rand(1, 1, 6).astype(np.float32),
        "right_hand": np.random.rand(1, 1, 6).astype(np.float32),
        "waist": np.random.rand(1, 1, 3).astype(np.float32),
    },
    "language": {"task": [["pick up the cup"]]}
}

request = {"endpoint": "get_action", "data": {"observation": observation}}
sock.send(msgpack.packb(request, default=encode_ndarray))
response = msgpack.unpackb(sock.recv(), raw=False, object_hook=decode_ndarray)

if isinstance(response, list):
    action = response[0]
    print("추론 성공! Action keys:", list(action.keys()))
    for key in action:
        print(f"  {key}: shape={np.array(action[key]).shape}")
elif isinstance(response, dict) and "error" in response:
    print("에러:", response["error"])
else:
    print("예상치 못한 응답:", type(response))

## 정리

base 모델이 응답하면 인프라·서빙 정상입니다. fine-tune 결과 검증은 모듈 8(closed-loop)에서 진행합니다.